# 🎯 AI Career Coach

## Learning Objectives
In this notebook, you will learn:
1. **PDF Resume Parsing** - Load and extract text from PDF resumes using PyPDFLoader
2. **System Prompt Engineering** - Design a career coach persona with context-aware prompting
3. **Conversational Memory** - Maintain multi-turn chat history with LangChain message types
4. **Streaming Responses** - Stream LLM output in real-time for interactive conversations

## Prerequisites
- Dependencies installed with `uv sync` in this project's folder
- `OPENAI_API_KEY` in this project's `.env` (the notebook uses `gpt-4o-mini` through `aied_helpers.get_openai_llm`)
- A candidate resume in PDF format at `content/Resume.pdf` (a sample ships with the project; replace it with yours)

---
## 📦 Step 1: Environment Setup

Load `OPENAI_API_KEY` from this project's `.env`, then create the chat model through the course's LLM factory.

In [ ]:
# ============================================================================
# ENVIRONMENT SETUP: Load API keys and suppress warnings
# ============================================================================
import warnings

from dotenv import find_dotenv, load_dotenv

warnings.filterwarnings("ignore")

# Reads OPENAI_API_KEY from this project's .env
load_dotenv(find_dotenv(usecwd=True))

print("✅ Environment ready")

In [ ]:
# ============================================================================
# LLM INITIALIZATION: The model behind the career coach
# ============================================================================
from aied_helpers import get_openai_llm

llm = get_openai_llm(model="gpt-4o-mini")

print(f"✅ LLM initialized: {llm.model_name}")

---
## 📄 Part 1: Resume Loading

Load the candidate's PDF resume and extract its text content for use as context.

In [ ]:
# ============================================================================
# RESUME LOADING: Extract Text from PDF
# ============================================================================
from langchain_community.document_loaders import PyPDFLoader

loader = PyPDFLoader("../content/Resume.pdf")
documents = loader.load()

context = "\n\n".join(doc.page_content for doc in documents)

print(f"📄 Loaded {len(documents)} page(s) from resume")

---
## 💬 Part 2: Career Coach System Prompt

Define the system message that shapes the LLM's persona as a professional career coach.
The resume content is injected directly into the system prompt so the coach has full context.

### Key Concepts:
- **SystemMessage** - Sets the AI's behavior and persona
- **HumanMessage / AIMessage** - Track the conversation turns
- **Chat History** - Maintains context across multiple exchanges

In [ ]:
# ============================================================================
# SYSTEM PROMPT: Define Career Coach Persona
# ============================================================================
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage

chat_history = []

system_message = SystemMessage(
    content=f"""
    You are a professional career coach and resume mentor.

    You help with:
    - Career Guidance
    - Resume Improvements
    - Interview Preparation
    - Job Search Strategy
    - Skill Gap Analysis

    Candidate Resume:
    {context}
    """
)

print("✅ Career coach system prompt configured")

---
## 🧠 Part 3: Conversational Memory in Action

`ask_coach()` runs one turn of the conversation: it appends the user's message to `chat_history`, sends the system prompt plus the whole history to the model, streams the reply token by token, and stores the reply in the history too. The second scripted question below says only *"those"* — the coach can answer it only because the first turn is still in `chat_history`.

In [ ]:
# ============================================================================
# CHAT TURN: Stream one reply and keep both sides in memory
# ============================================================================
def ask_coach(user_input: str) -> str:
    """Send one user message (plus the full history) and stream the coach's reply."""
    # Add user message to memory
    chat_history.append(HumanMessage(content=user_input))

    # Build message list: persona + resume first, then every turn so far
    messages = [system_message] + chat_history

    print("\n🤖 Coach:", end=" ", flush=True)

    response_text = ""

    for chunk in llm.stream(messages):
        print(chunk.content, end="", flush=True)
        response_text += chunk.content

    # Add AI response to memory
    chat_history.append(AIMessage(content=response_text))

    print("\n")
    return response_text


print("✅ ask_coach() ready")

In [ ]:
# ============================================================================
# SCRIPTED CONVERSATION: Two turns, the second one relies on memory
# ============================================================================
chat_history.clear()  # start from an empty conversation

for question in [
    "Name the two biggest gaps in my resume for a senior data scientist role. Keep it under 80 words.",
    "Give me one concrete step to close each of those in the next 3 months.",
]:
    print(f"You: {question}")
    ask_coach(question)

print(f"🧠 chat_history now holds {len(chat_history)} messages")

---
## 🚀 Part 4: Interactive Chat Loop (run by hand)

The loop below reads your questions with `input()`, so it is tagged `skip-execution` and does not run in automated runs. Run the cell yourself in Jupyter to chat with the coach; type `exit` or `quit` to end the session. It reuses `ask_coach()`, so the scripted turns above are still in memory.

In [ ]:
# ============================================================================
# INTERACTIVE CHAT: Streaming conversation loop (needs a person at the keyboard)
# ============================================================================
while True:
    user_input = input("You: ")

    if user_input.lower() in ["exit", "quit"]:
        print("\n👋 Goodbye!!")
        break

    ask_coach(user_input)

---
## 📝 Summary

In this notebook, we learned:

### 1. PDF Resume Parsing
- **PyPDFLoader** extracts text from multi-page PDF resumes for LLM consumption

### 2. System Prompt Engineering
- **Context injection** embeds the full resume into the system message for grounded responses

### 3. Conversational Memory
- **Chat history** with `HumanMessage` and `AIMessage` maintains multi-turn context — the scripted follow-up ("those") only makes sense because turn 1 is still in the history

### 4. Streaming Responses
- **`llm.stream()`** provides real-time token-by-token output for a responsive experience

### Next Steps
- **`Agent_Testing.ipynb`**: puts all four tools (checker, scorer, cover letter, coach) behind one LangGraph router and tests every routing path